In [ ]:
# ECE1512 Project A
# Sogol Samanian and Victor Mitea

# Code for SINGLE LAYER TEST. LoRA using SVD. The script loads the SmolVLM-256M (smallest variant) and pulls various layers from it

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForVision2Seq


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- Load model ---
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
model = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceTB/SmolVLM-256M-Instruct",
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2" if device.startswith("cuda") else "eager"
).to(device)
model.eval()

# ---- uncomment which set you want to test -----
# Q_proj layers
# vision_layers = [
#     'model.vision_model.encoder.layers.0.self_attn.q_proj',
#     'model.vision_model.encoder.layers.5.self_attn.q_proj',
#     'model.vision_model.encoder.layers.11.self_attn.q_proj',
# ]

# text_layers = [
#     'model.text_model.layers.0.self_attn.q_proj',
#     'model.text_model.layers.14.self_attn.q_proj',
#     'model.text_model.layers.27.self_attn.q_proj',
# ]

# MLP up projection layers
vision_layers = [
    'model.vision_model.encoder.layers.0.mlp.fc1',
    'model.vision_model.encoder.layers.5.mlp.fc1',
    'model.vision_model.encoder.layers.11.mlp.fc1',
]

text_layers = [
    'model.text_model.layers.0.mlp.up_proj',
    'model.text_model.layers.14.mlp.up_proj',
    'model.text_model.layers.27.mlp.up_proj',
]

layer_names = vision_layers + text_layers
# list of ranks being tested
ranks = [10,50,90,130,170,210,250,290,330,400,500]
trials = 5 # number of trials to average over

# Storage
rel_errors_all = torch.zeros(len(layer_names), len(ranks), trials)
accuracies_all = torch.zeros(len(layer_names), len(ranks), trials)
flops_red = []
mem_red = []

# Loop through layers
for li, layer_name in enumerate(layer_names):
    layer = dict(model.named_modules())[layer_name]
    W_orig = layer.weight.data.to(device)
    D_out, D_in = W_orig.shape
    bias_orig = layer.bias.data.to(device) if layer.bias is not None else None

    print(layer_name)
    # compute svd using torch function
    W_fp32 = W_orig.to(torch.float32)
    U, S, Vt = torch.linalg.svd(W_fp32, full_matrices=False)

    # loop through ranks ebing tested
    for ri, rank in enumerate(ranks):
        print("rank: " + str(rank))

        # this block calculates flops and memory recuctions.
        if li == 0:
            # this is the number of flops after LoRA
            flops_lora = D_out * rank + rank * D_in
            mem_lora = D_out * rank + rank * D_in
            flops_red.append(100 * (1 - flops_lora / (D_out * D_in)))
            mem_red.append(100 * (1 - mem_lora / (D_out * D_in)))

        # looping through trails
        for t in range(trials):
            print(t)
            # random input x
            x = torch.randn(512, D_in, device=device, dtype=W_orig.dtype)
            y_orig = x @ W_orig.T # calcuate baseline output value on random input
            if bias_orig is not None:
                y_orig += bias_orig

            # train a simple classifier on the baseline output data
            num_classes = 10
            labels = torch.randint(0, num_classes, (x.shape[0],), device=device)
            classifier = nn.Linear(D_out, num_classes, device=device, dtype=W_orig.dtype)
            criterion = nn.CrossEntropyLoss()
            optim_clf = torch.optim.Adam(classifier.parameters(), lr=1e-3)

            # train classifier on baseline output for 50 steps
            for epoch in range(50):
                optim_clf.zero_grad()
                logits = classifier(y_orig)
                loss = criterion(logits, labels)
                loss.backward()
                optim_clf.step()

            # caluclating LoRA approximation using SVD matrices
            U_r = U[:, :rank] # U, S, and V are truncated to rank dimension
            S_r = S[:rank]
            Vt_r = Vt[:rank, :]

            sqrt_S = torch.sqrt(S_r)
            # computing the A and B matrices that approximate W
            A = (U_r * sqrt_S.unsqueeze(0)).to(W_orig.dtype)
            B = (sqrt_S.unsqueeze(1) * Vt_r).to(W_orig.dtype)

            y_lora = x @ (A @ B).T # output on random input x after LoRA is compute using SVD
            if bias_orig is not None:
                y_lora += bias_orig

            # calculating relative error and storing it
            rel_errors_all[li, ri, t] = torch.norm(y_lora - y_orig) / torch.norm(y_orig) * 100
            with torch.no_grad():
                logits_lora = classifier(y_lora)
                pred = logits_lora.argmax(dim=1)
                accuracies_all[li, ri, t] = (pred == labels).float().mean() * 100

# averaging results over the trial numebr
rel_errors_avg = rel_errors_all.mean(dim=2).tolist()
accuracies_avg = accuracies_all.mean(dim=2).tolist()

rel_errors_vision = rel_errors_avg[:trials]
accuracies_vision = accuracies_avg[:trials]
rel_errors_text = rel_errors_avg[trials:]
accuracies_text = accuracies_avg[trials:]

# Plotting
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("SVD LoRA: Recon. Accuracy, Classifier Accuracy, FLOPs & Memory reduction vs. Rank",
             fontsize=12, y=0.99)
y_range = (0, 110)
x_range = (0, 550)

# Convert relative error to reconstruction accuracy
recon_errors_vision = [[100 - e for e in layer] for layer in rel_errors_vision]
recon_errors_text = [[100 - e for e in layer] for layer in rel_errors_text]

for i in range(3):
  # vision layers
    ax = axes[i, 0]
    ax.plot(ranks, recon_errors_vision[i], marker='o', label="Reconstruction Accuracy (%)")
    ax.plot(ranks, accuracies_vision[i], marker='x', label="Classifier Accuracy (%)")
    ax.plot(ranks, flops_red, linestyle='--', label="FLOPs & Memory Reduction (%)")
    ax.set_title(vision_layers[i], fontsize=9)
    ax.set_xlabel("Rank")
    ax.set_ylabel("Percent")
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)
    ax.grid(True)
# LM layers
    ax = axes[i, 1]
    ax.plot(ranks, recon_errors_text[i], marker='o', label="Reconstruction Accuracy (%)")
    ax.plot(ranks, accuracies_text[i], marker='x', label="Classifier Accuracy (%)")
    ax.plot(ranks, flops_red, linestyle='--', label="FLOPs & Memory Reduction (%)")
    ax.set_title(text_layers[i], fontsize=9)
    ax.set_xlabel("Rank")
    ax.set_ylabel("Percent")
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)
    ax.grid(True)

    # Only top-right plot gets a legend
    if i == 0:
        ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.show()



In [ ]:
# Used for producing FLOPs & Memory Reduction plot

import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
d_in = 1024   # input dimension
d_out = 1024  # output dimension
ranks = np.arange(1, 512, 8)  # rank values to test

# --- Theoretical FLOPs/memory % of baseline ---
# For LoRA: cost ~ (d_in * r + d_out * r)
# For baseline: cost ~ (d_in * d_out)
# => % of baseline = 100 * (r * (d_in + d_out)) / (d_in * d_out)
percent_of_baseline = 100 * (ranks * (d_in + d_out)) / (d_in * d_out)

# --- Plot ---
plt.figure(figsize=(6, 4))
plt.plot(ranks, percent_of_baseline, color='green',linestyle='--',label="FLOPs / Memory (% of baseline)")
plt.xlabel("Rank")
plt.ylabel("Percent of baseline")
plt.title("Theoretical FLOPs/Memory (% of baseline) vs. Rank")
plt.grid(True)
plt.xlim(0, ranks.max())
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
# Mini-Transformer model test
# LoRA on all Linear Layers ONLY

import torch
import copy

import torch.nn as nn
import pandas as pd
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# load MNIST data
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
subset_idx = torch.arange(3000)  # subset of MNIST dataset
train_data = Subset(train_data, subset_idx)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
# test data
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=128)

# mini transformer model class
# model dimension of 64, 4 heads, 2 blocks
class TinyTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(28, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.fc = nn.Linear(d_model, 10)

    def forward(self, x):
        x = x.squeeze(1)
        x = self.proj(x)
        x = self.encoder(x)
        x = x.mean(1)
        return self.fc(x)

# Training loop
def train_model(model, loader, epochs=10):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    # loop through epochs
    for epoch in range(epochs):
        correct, total = 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = F.cross_entropy(out, labels)
            loss.backward() # backwards pass
            opt.step()
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        print(f"Epoch {epoch+1}: acc={correct/total*100:.2f}%")
    return correct / total # return training accuracy

# evaluation function
def eval_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# class for LoRA-converted linear layers
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8):
        super().__init__()
        W = linear.weight.data
        U, S, Vt = torch.linalg.svd(W, full_matrices=False)
        r = min(rank, S.shape[0])
        self.A = nn.Parameter((U[:, :r] * S[:r]))  # (out_features, r)
        self.B = nn.Parameter(Vt[:r, :])           # (r, in_features)
        self._bias = linear.bias

    @property
    def weight(self):
        return self.A @ self.B

    @property
    def bias(self):
        return self._bias

    def forward(self, x):
        return x @ self.weight.T + (self.bias if self.bias is not None else 0)

# function to apply lora via SVD to all linear layers
def apply_svd_lora(model, rank=8):
    # model passed through this function to create new instances of LoraLinear class that approximate layer weights W via SVD into A and B
    # default rank set to 8
    for name, module in list(model.named_children()):
        if isinstance(module, nn.Linear):
            setattr(model, name, LoRALinear(module, rank))
        else:
            apply_svd_lora(module, rank)

# ranks to be tested
ranks = [1, 2, 4, 8, 16, 32, 64]
results = []
EPOCHS = 30 # number of epochs for training

# first train the baseline model
base = TinyTransformer().to(device)
train_acc_base = train_model(base, train_loader, epochs=EPOCHS)  # smaller epochs for quick demo
test_acc_base = eval_model(base, test_loader)

# print training results
params_base = sum(p.numel() for p in base.parameters())
results.append({"Rank": "baseline", "Training Accuracy": train_acc_base, "Test Accuracy": test_acc_base, "Params": params_base/1e3})
print(f"Baseline training accuracy: {train_acc_base*100:.2f}%, Baseline test accuracy: {test_acc_base*100:.2f}%, Params: {params_base/1e3:.2f}K")

# loop through LoRA ranks
for r in ranks:
    print(r)
    lora_model = copy.deepcopy(base) # make a copy of the trained model.
    apply_svd_lora(lora_model, rank=r) # apply Lora to the model!

    train_acc = 0.0
    test_acc = eval_model(lora_model, test_loader) # get test accuracy

    params = sum(p.numel() for p in lora_model.parameters()) # calcualte the number of parameters in the model

    results.append({"Rank": r, "Training Accuracy": train_acc, "Test Accuracy": test_acc, "Params": params/1e3})
    print(f"Rank {r}: training accuracy:{train_acc*100:.2f}% test accuracy: {test_acc*100:.2f}%, Params: {params/1e3:.2f}K")

df = pd.DataFrame(results)
print(df)

# plotting
fig, ax1 = plt.subplots(figsize=(8,5))

x_labels = [str(r["Rank"]) for r in results]
x = range(len(results))

# Accuracy bar plot (left y-axis)
ax1.bar(x, [r["Test Accuracy"]*100 for r in results], label='Test Accuracy')
ax1.set_ylabel("Test Accuracy (%)", color='blue')
ax1.set_xticks(x)
ax1.set_xticklabels(x_labels)
ax1.tick_params(axis='y', labelcolor='blue')

# Param number  line plot (right y-axis)
ax2 = ax1.twinx()
ax2.plot(x, [r["Params"] for r in results], marker='o', label='Params (K)', color='red')
ax2.set_ylabel("Params (K)", color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax1.set_ylim(0, 90)
ax2.set_ylim(0, 100)

plt.title("Test Accuracy & Parameters vs. Rank for SVD-LoRA on Linear Layers")
fig.tight_layout()
plt.show()


In [ ]:
# Mini-Transformer model test
# LoRA on KVQ Layers only

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import pandas as pd
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the MNIST data
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
subset_idx = torch.arange(3000) # subset of the data
train_data = Subset(train_data, subset_idx)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)


# test data
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=128)

# class for a normal linear layer that can switch to SVD approximation back and forth
class SVDFactorizedLinear(nn.Module):
  # during inference, can set_low_rank to get LoRA approximation of desired matrices
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

        self.low_rank_mode = False
        self.A = None
        self.B = None
        self.rank = None

    def set_low_rank(self, rank: int):
        # Compute SVD of current weight and store A,B
        with torch.no_grad():
            W = self.weight.data
            # SVD computation
            U, S, Vt = torch.linalg.svd(W, full_matrices=False)

            r = min(rank, S.shape[0])
            A = U[:, :r] * S[:r]        # compute A matrix from SVD results
            B = Vt[:r, :]               # compute B matrix from SVD results

            self.A = A
            self.B = B
            self.rank = r
            self.low_rank_mode = True
    # method to undo the above function
    def unset_low_rank(self):
        self.low_rank_mode = False
        self.A, self.B, self.rank = None, None, None # clear low rank representations

    def forward(self, x):
        if self.low_rank_mode and self.A is not None and self.B is not None:
            # Low-rank forward: x @ (B.T @ A.T)
            out = x @ self.B.T @ self.A.T
        else:
            out = x @ self.weight.T
        if self.bias is not None:
            out = out + self.bias
        return out

# custom multiheaded attention class to enable switching of KVQ to low rank
class SVDAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim # 64
        self.num_heads = num_heads # 4
        self.head_dim = embed_dim // num_heads

        # K V Q are full rank. Instantiated as SVD linear class to enable decomposition into A B during inference
        self.q_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)
        self.k_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)
        self.v_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)

        # Output projection stays full rank
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    # helper method to set low rank for all KVQ matrices
    def set_low_rank_qkv(self, rank: int):
        self.q_proj.set_low_rank(rank)
        self.k_proj.set_low_rank(rank)

        self.v_proj.set_low_rank(rank)

    # used to undo the above method
    def unset_low_rank_qkv(self):

        self.q_proj.unset_low_rank()
        self.k_proj.unset_low_rank()
        self.v_proj.unset_low_rank()

    def forward(self, x): # forward pass with custom attention computation.
        B, N, C = x.shape

        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(out)

# transformer block that incoroportates the custom attention class above
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4):
        super().__init__()
        self.attn = SVDAttention(embed_dim, num_heads) # initialized as the custom attention module
        self.norm1 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim),
        ) # defining the feed forward layer
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

# final class to incoroporate everything togething into full model
class RowTransformer(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, num_blocks=2):
        super().__init__()
        # Each row has 28 pixels and are project to embed_dim = 64
        self.row_embed = nn.Linear(28, embed_dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(embed_dim, 10)

    def forward(self, x):
        B = x.size(0)
        rows = x.squeeze(1)  # remove channel → (B,28,28)
        # embed each row → (B,28,embed_dim)
        x = self.row_embed(rows)
        # transformer blocks
        for blk in self.blocks:
            x = blk(x)
        # pool across rows
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits

    # finally, set low rank helper method to set across all classes.
    def set_low_rank_qkv(self, rank: int):
        for blk in self.blocks:
            blk.attn.set_low_rank_qkv(rank)
    def unset_low_rank_qkv(self):
        for blk in self.blocks:
            blk.attn.unset_low_rank_qkv()

# training loop
def train_model(model, loader, epochs=20, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    # loop over epochs
    for epoch in range(epochs):
        correct, total = 0, 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {correct/total*100:.2f}%")
    return correct / total # return training accuracy

# evaluate model
@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    correct, total = 0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return correct / total # return test accuracy

def count_effective_params(model):
    total = 0
    for module in model.modules():
        if isinstance(module, SVDFactorizedLinear) and module.low_rank_mode:
            # Count A and B (low-rank factors) + bias
            total += module.A.numel() + module.B.numel()
            if module.bias is not None:
                total += module.bias.numel()
        elif isinstance(module, SVDFactorizedLinear):
            # Count full weight + bias
            total += module.weight.numel()
            if module.bias is not None:
                total += module.bias.numel()
        elif isinstance(module, nn.Linear):
            total += module.weight.numel()
            if module.bias is not None:
                total += module.bias.numel()
    return total


# --- 7. Experiment: full-rank training, low-rank Q/K/V only at inference ---
def run_experiment():
    results = []
    # train full rank model
    model = RowTransformer().to(device)
    train_acc = train_model(model, train_loader, epochs=30)
    # evaluate baseline
    base_acc = eval_model(model, test_loader)
    print(f"Baseline Test Accuracy: {base_acc*100:.2f}%")
    params_base = count_effective_params(model)
    results.append({"Rank": "baseline", "TestAcc": base_acc, "Params(K)": params_base/1e3})

    # ranks to be tested
    ranks = [1, 2, 4, 8, 16, 32, 64]
    # loop over ranks
    for r in ranks:
        # set kvq to low rank mode
        model.set_low_rank_qkv(rank=r)
        # evaluate the model on test set
        acc = eval_model(model, test_loader)
        params = count_effective_params(model) # count the number of parameters
        print(f"Rank {r} → Test Accuracy: {acc*100:.2f}%, Params: {params/1e3:.2f}K")
        results.append({"Rank": r, "TestAcc": acc, "Params(K)": params/1e3})
        model.unset_low_rank_qkv()

    df = pd.DataFrame(results)
    print(df)

    # plot results
    fig, ax1 = plt.subplots(figsize=(8,5))

    x_labels = [str(r["Rank"]) for r in results]
    x = range(len(results))

    # Accuracy (left y-axis)
    ax1.bar(x, [r["TestAcc"]*100 for r in results], label='Test Accuracy')
    ax1.set_ylabel("Test Accuracy (%)", color='blue')
    ax1.set_xticks(x)
    ax1.set_xticklabels(x_labels)
    ax1.tick_params(axis='y', labelcolor='blue')

    # Params (right y-axis)
    ax2 = ax1.twinx()
    ax2.plot(x, [r["Params(K)"] for r in results], marker='o', label='Params (K)', color='red')
    ax2.set_ylabel("Params (K)", color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax1.set_ylim(0, 90)
    ax2.set_ylim(0, 100)

    plt.title("Test Accuracy & Parameters vs. Rank for SVD-LoRA on K,V,Q Layers")
    fig.tight_layout()
    plt.show()

    return df

if __name__ == "__main__":
    run_experiment()


In [ ]:
# mini-transfer model
# LoRA on KVQ and Linear Layers. Code combine the previous two sections to apply LoRA to the entire model.

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load MNIST data
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
subset_idx = torch.arange(3000)  # smaller subset
train_data = Subset(train_data, subset_idx)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

# test
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=128)

# class for a normal linear layer that can switch to SVD approximation back and forth
class SVDFactorizedLinear(nn.Module):
  # during inference, can set_low_rank to get LoRA approximation of desired matrices
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

        self.low_rank_mode = False
        self.A = None
        self.B = None
        self.rank = None

    def set_low_rank(self, rank: int):
        # Compute SVD of current weight and store A,B
        with torch.no_grad():
            W = self.weight.data
            # SVD computation
            U, S, Vt = torch.linalg.svd(W, full_matrices=False)

            r = min(rank, S.shape[0])
            A = U[:, :r] * S[:r]        # compute A matrix from SVD results
            B = Vt[:r, :]               # compute B matrix from SVD results

            self.A = A
            self.B = B
            self.rank = r
            self.low_rank_mode = True
    # method to undo the above function
    def unset_low_rank(self):
        self.low_rank_mode = False
        self.A, self.B, self.rank = None, None, None # clear low rank representations

    def forward(self, x):
        if self.low_rank_mode and self.A is not None and self.B is not None:
            # Low-rank forward: x @ (B.T @ A.T)
            out = x @ self.B.T @ self.A.T
        else:
            out = x @ self.weight.T
        if self.bias is not None:
            out = out + self.bias
        return out

# custom multiheaded attention class to enable switching of KVQ to low rank
class SVDAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim # 64
        self.num_heads = num_heads # 4
        self.head_dim = embed_dim // num_heads

        # K V Q are full rank. Instantiated as SVD linear class to enable decomposition into A B during inference
        self.q_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)
        self.k_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)
        self.v_proj = SVDFactorizedLinear(embed_dim, embed_dim, bias=True)

        # Output projection stays full rank
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    # helper method to set low rank for all KVQ matrices
    def set_low_rank_qkv(self, rank: int):
        self.q_proj.set_low_rank(rank)
        self.k_proj.set_low_rank(rank)

        self.v_proj.set_low_rank(rank)

    # used to undo the above method
    def unset_low_rank_qkv(self):

        self.q_proj.unset_low_rank()
        self.k_proj.unset_low_rank()
        self.v_proj.unset_low_rank()

    def forward(self, x): # forward pass with custom attention computation.
        B, N, C = x.shape

        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(out)

# transformer block that incoroportates the custom attention class above
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4):
        super().__init__()
        self.attn = SVDAttention(embed_dim, num_heads) # initialized as the custom attention module
        self.norm1 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim),
        ) # defining the feed forward layer
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

# final class to incoroporate everything togething into full model
class RowTransformer(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, num_blocks=2):
        super().__init__()
        # Each row has 28 pixels and are project to embed_dim = 64
        self.row_embed = nn.Linear(28, embed_dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(embed_dim, 10)

    def forward(self, x):
        B = x.size(0)
        rows = x.squeeze(1)  # remove channel → (B,28,28)
        # embed each row → (B,28,embed_dim)
        x = self.row_embed(rows)
        # transformer blocks
        for blk in self.blocks:
            x = blk(x)
        # pool across rows
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits

    # finally, set low rank helper method to set across all classes.
    def set_low_rank_qkv(self, rank: int):
        for blk in self.blocks:
            blk.attn.set_low_rank_qkv(rank)
    def unset_low_rank_qkv(self):
        for blk in self.blocks:
            blk.attn.unset_low_rank_qkv()

    # Convenience to toggle low-rank at inference across all blocks
    def set_low_rank_qkv(self, rank: int):
        for blk in self.blocks:
            blk.attn.set_low_rank_qkv(rank)

    def unset_low_rank_qkv(self):
        for blk in self.blocks:
            blk.attn.unset_low_rank_qkv()

# class for LoRA-converted linear layers
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8):
        super().__init__()
        W = linear.weight.data
        U, S, Vt = torch.linalg.svd(W, full_matrices=False)
        r = min(rank, S.shape[0])
        self.A = nn.Parameter((U[:, :r] * S[:r]))  # (out_features, r)
        self.B = nn.Parameter(Vt[:r, :])           # (r, in_features)
        self._bias = linear.bias

    @property
    def weight(self):
        return self.A @ self.B

    @property
    def bias(self):
        return self._bias

    def forward(self, x):
        return x @ self.weight.T + (self.bias if self.bias is not None else 0)

# function to apply lora via SVD to all linear layers
def apply_svd_lora(model, rank=8):
    # model passed through this function to create new instances of LoraLinear class that approximate layer weights W via SVD into A and B
    # default rank set to 8
    for name, module in list(model.named_children()):
        if isinstance(module, nn.Linear):
            setattr(model, name, LoRALinear(module, rank))
        else:
            apply_svd_lora(module, rank)

# training loop
def train_model(model, loader, epochs=20, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        correct, total = 0, 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {correct/total*100:.2f}%")
    return correct / total

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    correct, total = 0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return correct / total

# function for counting number of parameters
def count_effective_params(model):
    total = 0
    for module in model.modules():
        if isinstance(module, SVDFactorizedLinear) and module.low_rank_mode:
            total += module.A.numel() + module.B.numel()
            if module.bias is not None:
                total += module.bias.numel()
        elif isinstance(module, SVDFactorizedLinear):
            total += module.weight.numel()
            if module.bias is not None:
                total += module.bias.numel()
        elif isinstance(module, nn.Linear):
            total += module.weight.numel()
            if module.bias is not None:
                total += module.bias.numel()
    return total


# --- 7. Experiment: full-rank training, low-rank Q/K/V only at inference ---
def run_experiment():
    results = []
    # train the full rank baseline model
    model = RowTransformer().to(device)
    train_acc = train_model(model, train_loader, epochs=30)
    # evaluate the model and get the baseline test accuracy
    base_acc = eval_model(model, test_loader)
    print(f"Baseline Test Accuracy: {base_acc*100:.2f}%")
    params_base = count_effective_params(model) # count the number of baseline parameters
    results.append({"Rank": "baseline", "TestAcc": base_acc, "Params(K)": params_base/1e3})

    # ranks to be tested
    ranks = [1, 2, 4, 8, 16, 32, 64]
    # loop through ranks
    for r in ranks:
        lora_model = copy.deepcopy(model) # make a copy of the baseline model
        apply_svd_lora(lora_model, rank=r) # apply LoRA to LINEAR LAYERS
        lora_model.set_low_rank_qkv(rank=r) # apply LoRA to KVQ

        acc = eval_model(lora_model, test_loader) # get accuracy of LoRA model
        params = count_effective_params(lora_model) # count params of loRA model
        print(f"Rank {r} → Test Accuracy: {acc*100:.2f}%, Params: {params/1e3:.2f}K")
        results.append({"Rank": r, "TestAcc": acc, "Params(K)": params/1e3})
        model.unset_low_rank_qkv()

    df = pd.DataFrame(results)
    print(df)

    # plotting
    fig, ax1 = plt.subplots(figsize=(8,5))

    x_labels = [str(r["Rank"]) for r in results]
    x = range(len(results))

    # Accuracy (left y-axis)
    ax1.bar(x, [r["TestAcc"]*100 for r in results], label='Test Accuracy')
    ax1.set_ylabel("Test Accuracy (%)", color='blue')
    ax1.set_xticks(x)
    ax1.set_xticklabels(x_labels)
    ax1.tick_params(axis='y', labelcolor='blue')

    # Params (right y-axis)
    ax2 = ax1.twinx()
    ax2.plot(x, [r["Params(K)"] for r in results], marker='o', label='Params (K)', color='red')
    ax2.set_ylabel("Params (K)", color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax1.set_ylim(0, 90)     # first y-axis range
    ax2.set_ylim(0, 100)

    plt.title("Test Accuracy & Parameters vs. Rank for SVD-LoRA on All Layers")
    fig.tight_layout()
    plt.show()

    return df

if __name__ == "__main__":
    run_experiment()
